# BitVideo-1.58 Full-Scale Training on Kaggle## W1.58A8 Ternary Video Diffusion Transformer| Feature | Details ||---|---|| Model | BitVideo (dim=512, depth=8, ~50M params) || Weights | Ternary {-1, 0, +1} = 1.58 bits || Activations | INT8 per-token || Text Encoder | Flan-T5-Base (768D) || VAE | LTX-Video (128 latent channels) || Training | BF16 mixed precision, 8-bit AdamW || GPU | Kaggle T4 x2 (30GB) |### Setup1. Settings -> Accelerator -> GPU T4 x22. Turn on Internet3. Run All

In [ ]:
# Step 1: Environmentimport os, gc, sys, time, mathimport torchimport numpy as npprint(f'PyTorch: {torch.__version__}')print(f'GPU: {torch.cuda.get_device_name(0)}')print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

## Step 2: Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate safetensors sentencepiece!pip install -q opencv-python-headless imageio[ffmpeg] bitsandbytesprint('Done!')

## Step 3: Upload BitVideo PackageUpload your `bitvideo/` package folder as a Kaggle Dataset, or paste the key modules inline.

In [ ]:
# Option A: Clone from GitHub (if you push your repo)# !git clone https://github.com/YOUR_USER/bitvideo-158.git# sys.path.insert(0, 'bitvideo-158')# Option B: Inline minimal BitVideo for training# (The full package has 65 files - for Kaggle, we use the core modules)# For now, let's define the minimal training components inline:from torch.cuda.amp import GradScaler, autocastimport torch.nn as nnimport torch.nn.functional as Fprint('Ready')

## Step 4: Load Real Text Encoder

In [ ]:
from transformers import T5EncoderModel, T5Tokenizerprint('Loading Flan-T5-Base...')tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')text_encoder = T5EncoderModel.from_pretrained(    'google/flan-t5-base', torch_dtype=torch.float16).cuda().eval()for p in text_encoder.parameters():    p.requires_grad = FalseTEXT_DIM = text_encoder.config.d_modelprint(f'Text encoder: {TEXT_DIM}D')def encode_text(prompts, max_length=77):    tokens = tokenizer(prompts, return_tensors='pt', padding='max_length',                       truncation=True, max_length=max_length).to('cuda')    with torch.no_grad():        return text_encoder(**tokens).last_hidden_state.float()test = encode_text(['A dragon flying over mountains'])print(f'Test: {tuple(test.shape)}')del test; torch.cuda.empty_cache()

## Step 5: Load LTX Video VAE

In [ ]:
from diffusers.models import AutoencoderKLLTXVideoprint('Loading LTX Video VAE...')vae = AutoencoderKLLTXVideo.from_pretrained(    'Lightricks/LTX-Video', subfolder='vae', torch_dtype=torch.float16).cuda().eval()for p in vae.parameters():    p.requires_grad = Falseprint(f'VAE: {sum(p.numel() for p in vae.parameters()):,} params')# Verifywith torch.no_grad():    x = torch.randn(1, 3, 17, 128, 128, device='cuda', dtype=torch.float16)    z = vae.encode(x).latent_dist.sample()    y = vae.decode(z).sampleprint(f'Encode: {tuple(x.shape)} -> {tuple(z.shape)}')print(f'Decode: {tuple(z.shape)} -> {tuple(y.shape)}')LATENT_C = z.shape[1]del x, z, y; torch.cuda.empty_cache()

## Step 6: Prepare Video DatasetUpload your video clips as a Kaggle Dataset.Then encode them here with the VAE + text encoder.

In [ ]:
import cv2from pathlib import Path# Change this to your Kaggle dataset pathVIDEO_DIR = '/kaggle/input/your-video-dataset'ENCODED_DIR = '/kaggle/working/encoded'os.makedirs(f'{ENCODED_DIR}/latents', exist_ok=True)TARGET_FRAMES = 17TARGET_H, TARGET_W = 128, 128def encode_clip(video_path, caption):    cap = cv2.VideoCapture(str(video_path))    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))    if total <= 0: cap.release(); return None, None    indices = torch.linspace(0, total-1, TARGET_FRAMES).long().tolist()    frames = []    for idx in indices:        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)        ret, frame = cap.read()        if ret:            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)            frame = cv2.resize(frame, (TARGET_W, TARGET_H))            frames.append(torch.from_numpy(frame).float() / 127.5 - 1.0)        elif frames: frames.append(frames[-1].clone())    cap.release()    if len(frames) < TARGET_FRAMES: return None, None    video = torch.stack(frames).permute(3,0,1,2).unsqueeze(0).cuda().half()    with torch.no_grad():        latent = vae.encode(video).latent_dist.sample().squeeze(0).cpu().float()        text_emb = encode_text([caption]).squeeze(0).cpu()    del video; torch.cuda.empty_cache()    return latent, text_emb# Encode all clipsif Path(VIDEO_DIR).exists():    clips = sorted(Path(VIDEO_DIR).rglob('*.mp4'))    print(f'Found {len(clips)} clips')    metadata = []    for i, clip in enumerate(clips):        caption = clip.stem.replace('_', ' ')        latent, text_emb = encode_clip(clip, caption)        if latent is not None:            torch.save(latent, f'{ENCODED_DIR}/latents/video_{i:04d}.pt')            torch.save(text_emb, f'{ENCODED_DIR}/latents/text_{i:04d}.pt')            metadata.append({'video': f'latents/video_{i:04d}.pt', 'text': f'latents/text_{i:04d}.pt'})        if (i+1) % 50 == 0: print(f'  {i+1}/{len(clips)}')    import json    with open(f'{ENCODED_DIR}/metadata.json', 'w') as f:        json.dump(metadata, f)    print(f'Encoded {len(metadata)} clips')else:    print(f'No dataset at {VIDEO_DIR}')    print('Upload your clips as a Kaggle Dataset first!')

## Step 7: Define BitVideo Model (Full Scale)

In [ ]:
# Full-scale BitVideo for Kaggle T4x2from bitvideo.models import VideoDiTmodel = VideoDiT(    in_channels=LATENT_C,  # 128 (LTX VAE)    dim=512,               # 512 for T4x2 (or 768 if fits)    depth=8,               # 8 blocks    num_heads=8,    context_dim=TEXT_DIM,  # 768 (T5)    patch_size=(1, 2, 2),    ffn_expansion_ratio=4.0,    qk_norm=True,    device='cuda',    dtype=torch.float32,).train()print(f'Model: {model.parameter_count():,} params')print(f'Ternary weight memory: {model.parameter_count() * 2 / 8 / 1e6:.1f} MB')print(f'FP16 weight memory: {model.parameter_count() * 2 / 1e6:.1f} MB')print(f'Compression: 16x')

## Step 8: Train!

In [ ]:
# Training configMAX_STEPS = 50000BATCH_SIZE = 1GRAD_ACCUM = 8LR = 2e-4WARMUP = 2000# 8-bit optimizertry:    import bitsandbytes as bnb    optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=LR, weight_decay=0.01)    print('8-bit AdamW')except:    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)def cosine_lr(step):    if step < WARMUP: return step / WARMUP    return 0.5 * (1 + math.cos(math.pi * (step - WARMUP) / (MAX_STEPS - WARMUP)))lr_sched = torch.optim.lr_scheduler.LambdaLR(optimizer, cosine_lr)from bitvideo.training.losses import DiffusionLossfrom bitvideo.pipeline.schedulers import DDIMSchedulernoise_sched = DDIMScheduler(num_train_steps=1000, prediction_type='epsilon')loss_fn = DiffusionLoss(prediction_type='epsilon', snr_gamma=5.0)scaler = GradScaler()print(f'Training: {MAX_STEPS} steps, eff. batch {BATCH_SIZE*GRAD_ACCUM}, LR {LR}')

In [ ]:
# Main training loopfrom torch.utils.data import Dataset, DataLoaderimport jsonclass EncodedDataset(Dataset):    def __init__(self, root):        self.root = Path(root)        with open(self.root / 'metadata.json', 'r') as f:            self.samples = json.load(f)    def __len__(self): return len(self.samples)    def __getitem__(self, idx):        s = self.samples[idx]        v = torch.load(self.root / s['video'], weights_only=True)        t = torch.load(self.root / s['text'], weights_only=True)        return {'video_latent': v, 'text_embedding': t}dataset = EncodedDataset(ENCODED_DIR)dl = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)print(f'Dataset: {len(dataset)} samples')model.train()global_step = 0running_loss = 0.0t0 = time.time()data_iter = iter(dl)for step in range(MAX_STEPS * GRAD_ACCUM):    try: batch = next(data_iter)    except StopIteration: data_iter = iter(dl); batch = next(data_iter)        video = batch['video_latent'].cuda()    text = batch['text_embedding'].cuda()    B = video.shape[0]    t_step = torch.randint(0, 1000, (B,), device='cuda')    noise = torch.randn_like(video)    noisy = noise_sched.add_noise(video, noise, t_step)        with autocast(device_type='cuda', dtype=torch.bfloat16):        pred = model(noisy, t_step.float(), text)        loss = loss_fn(pred, noise, timesteps=t_step,                      alphas_cumprod=noise_sched.alphas_cumprod.cuda()) / GRAD_ACCUM        scaler.scale(loss).backward()    running_loss += loss.item() * GRAD_ACCUM        if (step + 1) % GRAD_ACCUM == 0:        scaler.unscale_(optimizer)        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        scaler.step(optimizer)        scaler.update()        optimizer.zero_grad(set_to_none=True)        lr_sched.step()        global_step += 1                if global_step % 100 == 0:            avg = running_loss / 100            elapsed = time.time() - t0            print(f'Step {global_step} | Loss {avg:.4f} | LR {optimizer.param_groups[0]["lr"]:.2e} | {elapsed:.0f}s')            running_loss = 0.0                if global_step % 5000 == 0:            torch.save({'model_state_dict': model.state_dict(), 'step': global_step},                      f'/kaggle/working/bitvideo_{global_step}.pt')            print(f'  Saved checkpoint')print(f'Training complete! Step {global_step}')

## Step 9: Generate Real Video

In [ ]:
model.eval()model.pack_weights()prompt = 'A futuristic underwater city with bioluminescent creatures, cinematic 4K'text_emb = encode_text([prompt])scheduler = DPMPlusPlusScheduler(num_train_steps=1000, prediction_type='epsilon')pipe = BitVideoPipeline(model, scheduler, decoder=None)with torch.no_grad():    latents = pipe(text_emb, num_frames=3, height=4, width=4,                   num_inference_steps=25, guidance_scale=1.0, decode=False)    decoded = vae.decode(latents.half()).samplevideo = decoded[0].permute(1,2,3,0).cpu().float().numpy()video = ((video - video.min()) / (video.max() - video.min()) * 255).clip(0,255).astype(np.uint8)import imageiowriter = imageio.get_writer('/kaggle/working/output.mp4', fps=24)for frame in video:    writer.append_data(frame)writer.close()print(f'Generated: {video.shape[0]} frames @ {video.shape[2]}x{video.shape[1]}')print('Saved: /kaggle/working/output.mp4')

---## SummaryThis notebook trains BitVideo-1.58 at scale:- **Real text encoder** (Flan-T5) for prompt understanding- **Real VAE** (LTX-Video) for pixel-quality output- **128-channel latents** matching the VAE's native space- **50K steps** with 8-bit optimizer and BF16 computeFor production quality, train for 200K+ steps with more data.